# Training on Colab

Trains MammoSeg on one or more cross-validation folds on a Colab GPU, like
`05_kaggle_training.ipynb` on Kaggle. It mounts Google Drive, unpacks the processed
data from an archive there, installs the package and runs `mammo-seg train` for
each fold, with the runs on Drive. Research use only: this is not a medical device.

**Once per session:** *Runtime* → *Change runtime type* → *T4 GPU*, before running
anything: changing it later starts a new machine. Drive asks for access when it is
mounted. Never put `kaggle.json` or any other credential in a notebook.

**The data** comes from an archive of the processed data of `DATASET` (`inbreast`
or `cbis-ddsm`), made once on a machine that has it (`mammo-seg manifest`,
`splits`, `masks` and `preprocess`, see the README) and uploaded to `DRIVE_DIR` as
`mammo_seg_data/processed/<DATASET>/mammo_processed_1024.tar.gz` (one folder per
dataset, as in `data/processed/`). It is made with:

```bash
tar czf mammo_processed_1024.tar.gz -C data/processed/<DATASET> \
    manifest.json splits.json preprocessed
```

It holds no raw images, but it is still the dataset: keep the Drive folder private
and never share it (the INbreast license is unclear; CBIS-DDSM must be cited).
For CBIS-DDSM, where every image has a mass, the training draws every image once
per epoch (`--positive-fraction none`).

**Pretraining and then INbreast:** train on CBIS-DDSM with its own `RUN_NAME`, then
set `DATASET = "inbreast"`, a new `RUN_NAME` and `INIT_FROM` to a checkpoint of the
CBIS-DDSM run on Drive, such as `"mammo_seg_runs/cbis-1024/fold_0/best.pt"`: every
fold then starts from its weights. Both runs need the same architecture and encoder
in `TRAIN_OPTIONS`. The splits are made with a fixed
seed, so an archive of the same manifest has the same folds as the Kaggle notebooks;
the setup prints the hashes of the splits and of the preprocessing to check that.

**Sessions end** without warning: when idle, when the GPU quota is used up, and
after at most about 12 hours. The runs are written straight to Drive
(`DRIVE_DIR/mammo_seg_runs/RUN_NAME`), and training saves `last.pt` after every
epoch, so nothing is lost but the epoch in progress: run the cells again in a new
session and a fold with a `last.pt` is resumed, a finished fold returns at once.
To continue a run started on Kaggle, copy the `runs` folder of its output to that
Drive folder first.

In [ ]:
from pathlib import Path

DATASET = "inbreast"  # or "cbis-ddsm"
FOLDS = [0, 1, 2, 3, 4]
MAX_MINUTES = 3 * 60  # for all folds together; training stops cleanly before this
# More options of `mammo-seg train`, for example "--epochs 2 --batch-size 2".
TRAIN_OPTIONS = "--epochs 60 --batch-size 4 --samples-per-epoch 256 --patience 0"
DRIVE_DIR = Path("/content/drive/MyDrive/mammo_seg")  # the project folder on Drive
RUN_NAME = "whole-1024"  # the runs are in DRIVE_DIR / "mammo_seg_runs" / RUN_NAME
# A checkpoint in DRIVE_DIR to start every fold from, for example
# "mammo_seg_runs/cbis-1024/fold_0/best.pt"; None starts as TRAIN_OPTIONS say.
INIT_FROM = None
REPO_URL = "https://github.com/GiannisPapa97/mammo_seg.git"
REPO_REF = "main"  # branch or tag to install

## 1. Setup

Mounts Drive, unpacks the archive of `DATASET` into `/content` (once per session), checks that
it holds the manifest, the splits and the preprocessed images, and installs the
package (`.[cloud]` leaves the PyTorch of Colab, a GPU build, untouched). The
repository is cloned to `repo`, not `mammo_seg`: a folder of that name next to the
notebook would hide the installed package from Python. Every session appends its
time and the commit of its code to `code_version.txt` in the runs folder.

In [ ]:
import hashlib
import shutil

from google.colab import drive

drive.mount("/content/drive")

WORK = Path("/content")
REPO = WORK / "repo"
assert DATASET in ("inbreast", "cbis-ddsm"), f"unknown DATASET {DATASET!r}"
PROCESSED = WORK / "data" / "processed" / DATASET
MANIFEST = PROCESSED / "manifest.json"
SPLITS = PROCESSED / "splits.json"
PREPROCESSED = PROCESSED / "preprocessed"
RUNS = DRIVE_DIR / "mammo_seg_runs" / RUN_NAME
ARCHIVE = (
    DRIVE_DIR / "mammo_seg_data" / "processed" / DATASET / "mammo_processed_1024.tar.gz"
)
assert ARCHIVE.is_file(), f"upload the archive of the processed data to {ARCHIVE}"
START = DRIVE_DIR / INIT_FROM if INIT_FROM else None
assert START is None or START.is_file(), f"no checkpoint {START} to start from"

# Unpacked on the machine: reading thousands of files from Drive is slow.
if not (PREPROCESSED / "preprocessing.json").exists():
    PROCESSED.mkdir(parents=True, exist_ok=True)
    local_archive = WORK / ARCHIVE.name
    shutil.copy(ARCHIVE, local_archive)
    !tar xzf "{local_archive}" -C "{PROCESSED}"
for path in (MANIFEST, SPLITS, PREPROCESSED / "preprocessing.json"):
    assert path.exists(), f"{path.relative_to(PROCESSED)} is not in {ARCHIVE.name}"
# Runs that continue or are compared with another platform need the same files.
for path in (SPLITS, PREPROCESSED / "preprocessing.json"):
    print(f"{path.name}: sha256 {hashlib.sha256(path.read_bytes()).hexdigest()}")

!test -d {REPO} || git clone -q --branch {REPO_REF} {REPO_URL} {REPO}
!pip install -q "{REPO}[cloud]"
!nvidia-smi -L

# The code of every session: the time and the commit, kept with the runs.
RUNS.mkdir(parents=True, exist_ok=True)
VERSION = RUNS / "code_version.txt"
!echo "$(date -u +%FT%TZ) $(git -C {REPO} rev-parse HEAD)" | tee -a "{VERSION}"

## 2. Train

One run per fold in `FOLDS`, in `fold_<k>` of the runs folder: `best.pt`,
`last.pt` and `history.jsonl`. `MAX_MINUTES` is shared by the folds: each gets what
is left. `--device cuda` makes a session without a GPU an error instead of a slow
run on the CPU. The options of the dataset come before `TRAIN_OPTIONS`, which can
change them; `INIT_FROM` is passed on every session, as a run is resumed only with
the checkpoint it started from.

In [ ]:
import time

# CBIS-DDSM has no image without a mass to balance against.
DATASET_OPTIONS = "--positive-fraction none" if DATASET == "cbis-ddsm" else ""
if START is not None:
    DATASET_OPTIONS += f' --init-from "{START}"'
started = time.monotonic()
for fold in FOLDS:
    minutes = int(MAX_MINUTES - (time.monotonic() - started) / 60)
    if minutes < 1:
        print(f"fold {fold}: MAX_MINUTES is used up; run this cell in a new session")
        continue
    resume = "--resume" if (RUNS / f"fold_{fold}" / "last.pt").exists() else ""
    print(f"=== fold {fold}: {'resuming' if resume else 'new run'}, {minutes} min left")
    !mammo-seg train --manifest {MANIFEST} --splits {SPLITS} \
        --preprocessed-dir {PREPROCESSED} --output-dir "{RUNS}" --device cuda \
        --num-workers 2 {DATASET_OPTIONS} {TRAIN_OPTIONS} --fold {fold} \
        --max-minutes {minutes} {resume}

## 3. Results

The best epoch of each fold by the selection metric (Dice over all validation
images unless changed), with the Dice on the images with a mass, their sensitivity
and the share of images without a mass on which anything was predicted. `n/a` means
the score was undefined. The last line writes everything to Drive.

In [ ]:
import json
import statistics


def show(value):
    return "n/a" if value is None else f"{value:.3f}"


selected = []
for fold in FOLDS:
    history = RUNS / f"fold_{fold}" / "history.jsonl"
    lines = history.read_text().splitlines() if history.exists() else []
    rows = [json.loads(line) for line in lines]
    scored = [row for row in rows if row["selection_value"] is not None]
    if not scored:
        print(f"fold {fold}: {len(rows)} epochs, none with a defined score")
        continue
    best = max(scored, key=lambda row: row["selection_value"])
    validation = best["validation"]
    selected.append(best["selection_value"])
    print(
        f"fold {fold}: {len(rows)} epochs, best epoch {best['epoch']}: "
        f"score {show(best['selection_value'])}, "
        f"Dice {show(validation['all_images']['dice'])}, "
        f"with mass {show(validation['with_mass']['dice'])} "
        f"(sensitivity {show(validation['with_mass']['sensitivity'])}), "
        f"false alarms {show(validation['without_mass']['false_alarm_rate'])}"
    )
    # What the epochs cost, to plan a larger input or batch.
    seconds = statistics.median(row["seconds"] for row in rows)
    peaks = [row.get("peak_gpu_memory_gb") for row in rows]
    peaks = [peak for peak in peaks if peak is not None]
    peak = f"{max(peaks):.1f} GB" if peaks else "not recorded"
    print(f"    {seconds:.0f} s per epoch (median), peak GPU memory {peak}")
if len(selected) > 1:
    print(f"mean score over {len(selected)} folds: {sum(selected) / len(selected):.3f}")

drive.flush_and_unmount()  # mount again (the setup cell) to use Drive afterwards